# BNP Causal Survival Estimation with Auxiliary Summary Information

This notebook develops the main extension of the project.

After reconstructing individual patient data (IPD) across multiple trials, naive pooling does not in general identify a meaningful causal survival estimand, because each trial represents a different underlying population.

To address this, we:
1. define a target population using auxiliary summary information
2. reweight the pooled reconstructed IPD toward that target
3. estimate treatment-specific survival probabilities at a fixed time point
4. quantify uncertainty using a Bayesian nonparametric weighting scheme

In [1]:
# For now, use simulated multitrial data as the reconstructed pooled dataset
import pandas as pd
import numpy as np

from src.simulation.trial_generator import simulate_multi_trial

multi = simulate_multi_trial(n_trials=4, n_per_trial=250, seed=733)
df_multi = multi.ipd.copy()

df_multi.head()

,age,biomarker,stage,treatment,time,event,trial_id
0,67.106302,1,2,0,1.778858,1,trial_001
1,55.792886,0,2,0,24.000000,0,trial_001
2,55.350177,0,3,0,0.116891,1,trial_001
3,60.424046,1,1,1,0.889068,0,trial_001
4,55.414515,0,1,1,3.516167,1,trial_001


## Target Estimand

We focus on the fixed-time causal survival contrast
$$
\Delta(t_0) = S^1(t_0) - S^0(t_0),
$$
where
$$
S^a(t_0) = \Pr(T^a > t_0)
$$
denotes the survival probability at time $t_0$ under treatment $a \in \{0,1\}$.

In the pooled setting, this estimand depends on the target population over which the counterfactual survival probabilities are defined.

In [2]:
t0 = 12.0

In [3]:
# compute the naive pooled estimate
from src.causal.estimands import pooled_survival_difference

naive = pooled_survival_difference(df_multi, t0=t0)
naive

,t0,S0,S1,Delta
0,12.0,0.403394,0.474549,0.071155


In [4]:
# Baseline covariate distribution
covariates = ["age", "biomarker", "stage"]

df_multi.groupby("trial_id")[covariates].mean()

,age,biomarker,stage
trial_id,,,
trial_001,60.596765,0.400,1.636
trial_002,56.125191,0.576,1.848
trial_003,58.726448,0.608,1.740
trial_004,53.194751,0.536,1.664


In [5]:
# For now, create a target summary vector. Later, this will represent Table 1-style reported summaries
target_means = pd.Series(
    {
        "age": 58.0,
        "biomarker": 0.55,
        "stage": 1.75,
    }
)

target_means

age          58.00
biomarker     0.55
stage         1.75
dtype: float64

## Auxiliary Summary Information

We treat the target population as partially observed through summary statistics, such as means or proportions reported in baseline tables.

In this prototype, the target population is defined by the target covariate means above. These quantities play the role of auxiliary summary information.

In [6]:
# Compute constrained weights
from src.causal.pooling import compute_covariate_means, reweight_to_target, weighted_survival_difference
w_target = reweight_to_target(
    df=df_multi,
    covariates=covariates,
    target_means=target_means,
)

weighted_means = compute_covariate_means(df_multi, covariates, weights=w_target)
weighted_means

/Users/jonathanma/Desktop/MSEnotes/733_NPBayes/FinalProject/NPBS-project-MaZhu/src/causal/pooling.py:129: RuntimeWarning: overflow encountered in matmul
  logits = X @ theta #softmax stabilization


age          57.999912
biomarker     0.549998
stage         1.749997
dtype: float64

In [7]:
weighted = weighted_survival_difference(
    df=df_multi,
    weights=w_target,
    t0=t0,
    random_state=733,
)
weighted

{'t0': 12.0,
 'S0': 0.4091253884480398,
 'S1': 0.45042317714363894,
 'Delta': 0.041297788695599136}